# Breast Cancer (BCSC) - Visão geral do dataset

Dataset: https://www.openml.org/d/43687 (dados do Breast Cancer Surveillance Consortium)  
Variável alvo: `Is_Binary_Indicator_Of_Cancer_Diagnosis` (câncer diagnosticado em até 1 ano após a mamografia)

In [ ]:
import pandas as pd

pd.set_option("display.max_columns", None)

df = pd.read_csv("../dataset/breast_cancer_screening_bcsc.csv")

print(f"Linhas: {df.shape[0]} | Colunas: {df.shape[1]}")
print(f"Mulheres distintas: {df['Patients_Study_ID'].nunique()}")
df.head()

## Tipos de dados e valores não nulos

In [ ]:
df.info()

## Valores ausentes e duplicados

In [ ]:
ausentes = pd.DataFrame({
    "ausentes": df.isna().sum(),
    "%": (df.isna().mean() * 100).round(1),
})
print(ausentes[ausentes["ausentes"] > 0])
print(f"\nLinhas duplicadas: {df.duplicated().sum()}")

## Colunas numéricas

In [ ]:
df[["Age_At_The_Time_Of_Mammography", "Body_Mass_Index"]].describe().T

## Colunas categóricas

In [ ]:
categoricas = df.select_dtypes(exclude="number").columns
for col in categoricas:
    print(f"--- {col} ---")
    print(df[col].value_counts(dropna=False).to_string(), "\n")

## Distribuição da variável alvo

Classes muito desbalanceadas: acurácia sozinha engana, vale olhar recall, precisão e AUC.

In [ ]:
alvo = "Is_Binary_Indicator_Of_Cancer_Diagnosis"
pd.DataFrame({
    "quantidade": df[alvo].value_counts(),
    "proporção (%)": (df[alvo].value_counts(normalize=True) * 100).round(2),
})

## Atenção: `Cancer_Type` vaza a resposta

Essa coluna só diz "No cancer diagnosis" quando o alvo é `False`, então deve ser removida antes de treinar.

In [ ]:
pd.crosstab(df["Cancer_Type"], df[alvo])

## Taxa de câncer por avaliação do radiologista

In [ ]:
(
    df.groupby("Radiologists_Assessment")[alvo]
    .agg(["count", "sum", "mean"])
    .rename(columns={"count": "exames", "sum": "cancer", "mean": "taxa"})
    .sort_values("taxa", ascending=False)
    .round(3)
)